# Wind-rose plots of synthetic deviated angles

This notebook applies the same 10-degree polar-histogram procedure used in the paper workflow to the public synthetic dataset. The values and condition labels are synthetic and are provided only to demonstrate the code.


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_FILENAME = "example_synthetic_deviated_angles.xlsx"
SHEET_NAME = "Deviated angles"

# Edit this list to plot any conditions present in the Excel workbook.
CONDITIONS = [
    "CONTROL",
    "TREATMENT_W",
    "TREATMENT_X",
    "TREATMENT_Y",
    "TREATMENT_Z",
]

# These bin limits reproduce the original analysis settings.
ANGLE_MIN = -90
ANGLE_MAX = 180
BIN_WIDTH = 10


In [ ]:
relative_dataset = Path("data") / DATA_FILENAME
candidates = [
    Path.cwd() / relative_dataset,
    Path.cwd().parent / relative_dataset,
    Path.cwd() / "public_repository" / relative_dataset,
    Path.cwd() / DATA_FILENAME,
]
DATA_FILE = next((path for path in candidates if path.exists()), None)

if DATA_FILE is None:
    checked = "\n".join(f"- {path.resolve()}" for path in candidates)
    raise FileNotFoundError(
        f"The included example dataset '{DATA_FILENAME}' was not found. Checked:\n{checked}"
    )

data = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)
data.columns = data.columns.str.strip()

missing = [condition for condition in CONDITIONS if condition not in data.columns]
if missing:
    raise KeyError(f"Conditions not found in the workbook: {missing}")

print("Dataset:", DATA_FILE.resolve())
print("Conditions to plot:", CONDITIONS)


In [ ]:
def safe_filename(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_").lower()


def plot_wind_rose(theta_degrees, condition, output_directory=None):
    values = pd.to_numeric(theta_degrees, errors="coerce").dropna().to_numpy()
    if values.size == 0:
        raise ValueError(f"No numeric angles were found for {condition}.")

    outside = values[(values < ANGLE_MIN) | (values > ANGLE_MAX)]
    if outside.size:
        raise ValueError(
            f"{condition} contains angles outside [{ANGLE_MIN}, {ANGLE_MAX}]: "
            f"{outside.tolist()}"
        )

    edges_deg = np.arange(ANGLE_MIN, ANGLE_MAX + BIN_WIDTH, BIN_WIDTH)
    edges_rad = np.deg2rad(edges_deg)
    counts, _ = np.histogram(values, bins=edges_deg)

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"})
    ax.bar(
        edges_rad[:-1],
        counts,
        width=np.deg2rad(BIN_WIDTH),
        align="edge",
        color="#AA4499",
        edgecolor="black",
        linewidth=1.0,
        alpha=0.9,
    )

    # Set 0 degrees at the top and increase angles clockwise.
    ax.set_theta_direction(-1)
    ax.set_theta_offset(np.pi / 2.0)
    tick_deg = np.arange(0, 360, 30)
    ax.set_xticks(np.deg2rad(tick_deg))
    ax.set_xticklabels([f"{degree}°" if degree <= 180 else f"-{360-degree}°" for degree in tick_deg])
    ax.set_title(f"{condition} (n = {values.size})", pad=20)
    fig.set_dpi(300)

    if output_directory is not None:
        output_directory.mkdir(parents=True, exist_ok=True)
        stem = f"wind_rose_{safe_filename(condition)}"
        fig.savefig(output_directory / f"{stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(output_directory / f"{stem}.tiff", dpi=300, bbox_inches="tight")

    return fig, ax, counts


In [ ]:
OUTPUT_DIRECTORY = Path("outputs") / "wind_rose"

for condition in CONDITIONS:
    figure, axis, bin_counts = plot_wind_rose(
        data[condition],
        condition,
        output_directory=OUTPUT_DIRECTORY,
    )
    plt.show()

print(f"PNG and TIFF files were saved in: {OUTPUT_DIRECTORY.resolve()}")


## Plotting different conditions

Edit only `CONDITIONS` in the configuration cell. Names must match the Excel column headers exactly. For example:

```python
CONDITIONS = ["CONTROL", "TREATMENT_Z"]
```

To use another workbook, place it in `data/`, update `DATA_FILENAME`, and update `SHEET_NAME` if needed. Each condition must occupy one column, with one deviated angle in degrees per cell.
